# XGBoost Add-on Experiment + Comparison (open-to-open vs close-to-close)

This notebook runs the pipeline with **XGBoost enabled** and then visualizes performance across:
- horizons (1D vs 3D)
- models (logistic_regression vs random_forest vs xgboost)
- feature sets (F1–F5 ladder)
- execution modes (open_to_open vs close_to_close)

If `xgboost` is not installed, install it first:
```bash
pip install xgboost
```


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.config import AppConfig
from src.pipeline import run_pipeline

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)


## 1) Run pipeline with XGBoost enabled
We run with a new notebook tag so outputs don't overwrite your previous run.

In [ ]:
config = AppConfig()
config.use_xgboost = True

# Keep your current ladder feature sets (F5 excludes fundamentals)
print('Feature sets:', list(config.feature_sets.keys()))
print('Horizons:', config.horizons)
print('Models: logistic_regression, random_forest, xgboost')

notebook_tag = '08'
mode = 'live'  # change to 'synthetic' for fast debug

paths = run_pipeline(notebook_tag=notebook_tag, mode=mode, config=config)
paths

## 2) Load outputs

In [ ]:
metrics = pd.read_csv(paths['metrics'])
bt = pd.read_csv(paths['backtest'])
rel = pd.read_csv(paths['relative'])

KEY = ['feature_set','model','horizon_days']
df = bt.merge(rel, on=KEY+['execution_mode'], how='left')
df = df.merge(metrics, on=KEY, how='left', suffixes=('','_ml'))

print('metrics rows:', len(metrics))
print('backtest rows:', len(bt))
print('merged rows:', len(df))
df[['feature_set','model','horizon_days','execution_mode','sharpe','annualized_return','max_drawdown','alpha_ann','information_ratio','balanced_accuracy','rank_ic']].sort_values('sharpe', ascending=False).head(15)

## 3) Heatmap grids: Sharpe landscape
For each execution_mode + horizon, show a feature_set × model heatmap.

In [ ]:
def heatmap(pivot, title):
    if pivot.empty: return
    plt.figure(figsize=(10,4))
    plt.imshow(pivot.values, aspect='auto')
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=45, ha='right')
    plt.yticks(range(len(pivot.index)), pivot.index)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            v = pivot.iloc[i,j]
            s = 'NA' if pd.isna(v) else f'{v:.2f}'
            plt.text(j,i,s,ha='center',va='center',fontsize=8)
    plt.colorbar()
    plt.title(title)
    plt.tight_layout()
    plt.show()

for em in sorted(df.execution_mode.unique()):
    for h in sorted(df.horizon_days.unique()):
        sub = df[(df.execution_mode==em) & (df.horizon_days==h)]
        p = sub.pivot_table(index='feature_set', columns='model', values='sharpe', aggfunc='first')
        heatmap(p, f'Sharpe heatmap | {em} | h={h}D')

## 4) Top-N table per (execution_mode, horizon)

In [ ]:
TOP_N = 5
for em in sorted(df.execution_mode.unique()):
    for h in sorted(df.horizon_days.unique()):
        sub = df[(df.execution_mode==em) & (df.horizon_days==h)].sort_values('sharpe', ascending=False).head(TOP_N)
        print('=== Top',TOP_N,'|',em,'| h=',h,'===')
        display(sub[['feature_set','model','sharpe','annualized_return','max_drawdown','alpha_ann','information_ratio','avg_turnover','avg_cost_drag','balanced_accuracy','rank_ic']])

## 5) Historical equity curves for selected strategies (per execution_mode)
Loads curve CSVs saved by the pipeline under `outputs/curves/`.

In [ ]:
ROOT = Path(r'C:/Users/user/Downloads/Moltbot/HKU-FYP/fyp_finance_ml_v2')
OUT = ROOT / 'outputs'
curves_dir = OUT / 'curves'

def load_curve(path: Path):
    d = pd.read_csv(path)
    d['date'] = pd.to_datetime(d['date'])
    d = d.sort_values('date')
    d['equity'] = (1 + d['net_ret']).cumprod()
    return d

def curve_name(fs, model, h, em):
    return f'{notebook_tag}_{mode}_{fs}_{model}_{int(h)}d_{em}_daily.csv'

# plot top-3 strategies per exec_mode and horizon
for em in sorted(df.execution_mode.unique()):
    for h in sorted(df.horizon_days.unique()):
        sub = df[(df.execution_mode==em) & (df.horizon_days==h)].sort_values('sharpe', ascending=False).head(3)
        plt.figure(figsize=(10,4))
        spy_path = curves_dir / f'{notebook_tag}_{mode}_SPY_{int(h)}d_daily.csv'
        if spy_path.exists():
            spy = load_curve(spy_path)
            plt.plot(spy['date'], spy['equity'], label='SPY', linewidth=2)

        for _, r in sub.iterrows():
            p = curves_dir / curve_name(r['feature_set'], r['model'], r['horizon_days'], em)
            if p.exists():
                c = load_curve(p)
                plt.plot(c['date'], c['equity'], label=f'{r["feature_set"]}|{r["model"]}')

        plt.title(f'Top-3 equity curves | {em} | h={h}D')
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.show()